In [1]:
import editor

import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# settings
array_size = 512
num_channels = 3

nn_intake_fov = 15
nn_hidden_dim = 10

run_device = torch.device("cuda")

In [3]:
class Net(nn.Module):
    def __init__(self, kernel_size: int):
        super().__init__()
        self.conv1 = nn.Conv2d(num_channels, 6, 7)
        self.conv2 = nn.Conv2d(6, 9, 5)
        self.conv3 = nn.Conv2d(9, 12, 3)
        self.conv4 = nn.Conv2d(12, 12, 3)

        self.fc1 = nn.Linear(12, num_channels);          self.bn1 = nn.BatchNorm1d(num_channels)
        # self.fc2 = nn.Linear(nn_hidden_dim, nn_hidden_dim);  self.bn2 = nn.BatchNorm1d(nn_hidden_dim)
        # self.fc3 = nn.Linear(nn_hidden_dim, num_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))

        x = x.flatten(start_dim=1)

        # print(x.size())

        # x = F.relu(self.bn1(self.fc1(x)))
        # x = F.relu(self.bn2(self.fc2(x)))
        x = self.bn1(self.fc1(x))

        return x

In [4]:
class TempSim:
    def __init__(self, model):
        self.model = model
        
    def extract_kernel(self, global_tensor: torch.tensor, start_x, end_x, start_y, end_y):
        return global_tensor[:, start_x:end_x, start_y:end_y]
    
    def step(self, state: torch.tensor):
        state = state.to(run_device)
        
        C, H, W = state.shape
        k = nn_intake_fov
        lo, hi = (k - 1) // 2, k // 2
        x = F.pad(state[None], (lo, hi, lo, hi), mode="circular")
        slices = F.unfold(x, k).transpose(1, 2).reshape(H * W, C, k, k)  # one slice per cell
        out = self.model(slices)                                         # (H*W, outputs per slice)
        
        out = out.cpu()
        
        return out.reshape(H, W, -1).permute(2, 0, 1)                    # (outputs per slice, H, W)

In [5]:
model = Net(nn_intake_fov).to(run_device)

In [6]:
sim = TempSim(model)

In [7]:
editor = editor.NCAEditor(sim, torch.rand((num_channels, array_size, array_size)))

In [ ]:
editor.run()

In [ ]:
def extract_kernel(global_tensor: torch.tensor, start_x, end_x, start_y, end_y):
    return global_tensor[:, start_x:end_x, start_y:end_y]

In [ ]:
startx = 0
starty = 0

endx = 2
endy = 2

In [ ]:
x = torch.tensor(((1, 2, 3, 4),
                  (5, 6, 7, 8),
                  (9, 0, 1, 2),
                  (3, 4, 5, 6))).unsqueeze(0)

In [ ]:
extract_kernel(x, 1, 4, 1, 4)

tensor([[[6, 7, 8],
         [0, 1, 2],
         [4, 5, 6]]])